GOOGLE COLAB – KUIS SBP (SAW, EDAS, TOPSIS)

🔹 1. IMPORT LIBRARY

In [1]:
import pandas as pd
import numpy as np

🔹 2. LOAD DATASET

Upload dulu file:

Data_Polis.csv
Data_Klaim.csv

In [22]:
from google.colab import files
uploaded = files.upload()

df_polis = pd.read_csv('Data_Polis.csv')
df_klaim = pd.read_csv('Data_Klaim.csv')

Saving Data_Polis.csv to Data_Polis (1).csv
Saving Data_Klaim.csv to Data_Klaim (1).csv


🔹 3. PREPROCESSING (WAJIB)

In [3]:
df_polis.columns = df_polis.columns.str.strip().str.lower().str.replace(' ', '_')
df_klaim.columns = df_klaim.columns.str.strip().str.lower().str.replace(' ', '_')

df = pd.merge(df_polis, df_klaim, on='nomor_polis')
df.head()

,nomor_polis,plan_code,gender,tanggal_lahir,tanggal_efektif_polis,domisili,claim_id,reimburse/cashless,inpatient/outpatient,icd_diagnosis,icd_description,status_klaim,tanggal_pembayaran_klaim,tanggal_pasien_masuk_rs,tanggal_pasien_keluar_rs,nominal_klaim_yang_disetujui,nominal_biaya_rs_yang_terjadi,lokasi_rs
0,POL-0003,M-001,M,19790821,20160808,JAKARTA,C-3535-M,C,IP,S83.2,"TEAR OF MENISCUS, CURRENT",PAID,2024-10-31,2024-09-09,2024-09-10,14138163.0,14138163.0,Indonesia
1,POL-0006,M-001,F,19551127,20160628,JAKARTA,C-4498-M,C,IP,S46.0,INJURY OF MUSCLE(S) AND TENDON(S) OF THE ROTAT...,PAID,2024-11-13,2024-08-16,2024-08-22,117940836.0,120388776.0,Indonesia
2,POL-0006,M-001,F,19551127,20160628,JAKARTA,C-4499-M,C,IP,T84.7,INFECTION AND INFLAMMATORY REACTION DUE TO OTH...,PAID,2024-12-13,2024-10-23,2024-10-24,60671660.0,61800880.0,Indonesia
3,POL-0010,M-001,F,19570605,20170908,JAKARTA,C-3447-M,R,IP,K31.7,POLYP OF STOMACH AND DUODENUM,PAID,2024-02-27,2024-01-19,2024-01-19,1723747.0,1723747.0,Singapore
4,POL-0010,M-001,F,19570605,20170908,JAKARTA,C-3448-M,R,IP,K31.7,POLYP OF STOMACH AND DUODENUM,PAID,2024-02-27,2024-01-18,2024-01-18,63251531.0,63251531.0,Singapore


🔹 4. PEMBENTUKAN KRITERIA

In [4]:
df_clean = df.copy()

# status klaim → numerik
df_clean['status_klaim'] = df_clean['status_klaim'].astype(str)
df_clean['status_klaim'] = df_clean['status_klaim'].apply(
    lambda x: 1 if 'approve' in x.lower() else 0
)

# inpatient/outpatient → numerik
df_clean['inpatient_outpatient'] = df_clean['inpatient/outpatient'].astype(str)
df_clean['inpatient_outpatient'] = df_clean['inpatient_outpatient'].apply(
    lambda x: 2 if 'inpatient' in x.lower() else 1
)

# pilih kriteria
kriteria = [
    'nominal_klaim_yang_disetujui',
    'nominal_biaya_rs_yang_terjadi',
    'status_klaim',
    'inpatient_outpatient'
]

# ubah ke numerik
df_clean[kriteria] = df_clean[kriteria].apply(pd.to_numeric, errors='coerce')
df_clean = df_clean.dropna()

# bobot (SAMA untuk semua metode)
bobot = np.array([0.3, 0.3, 0.2, 0.2])

# jenis kriteria (False = cost, True = benefit)
benefit = [False, False, True, True]

METODE SAW

🔹 Normalisasi

In [5]:
df_saw = df_clean.copy()

for i, col in enumerate(kriteria):
    if benefit[i]:
        df_saw[col] = df_saw[col] / df_saw[col].max()
    else:
        df_saw[col] = df_saw[col].min() / df_saw[col]

🔹 Hitung skor

In [6]:
df_saw['SAW_score'] = df_saw[kriteria].values.dot(bobot)

ranking_saw = df_saw[['nomor_polis','SAW_score']].sort_values(by='SAW_score', ascending=False)
ranking_saw['Rank_SAW'] = range(1, len(ranking_saw)+1)

print("TOP 10 SAW")
ranking_saw.head(10)

TOP 10 SAW


,nomor_polis,SAW_score,Rank_SAW
0,POL-0003,NaN,1
1,POL-0006,NaN,2
2,POL-0006,NaN,3
3,POL-0010,NaN,4
4,POL-0010,NaN,5
5,POL-0011,NaN,6
6,POL-0011,NaN,7
7,POL-0013,NaN,8
8,POL-0018,NaN,9
9,POL-0020,NaN,10


METODE EDAS

🔹 Average

In [9]:
avg = df_clean[kriteria].mean()

🔹 PDA & NDA

In [8]:
PDA = pd.DataFrame()
NDA = pd.DataFrame()

for i, col in enumerate(kriteria):
    if benefit[i]:
        PDA[col] = np.maximum(0, df_clean[col] - avg[col])
        NDA[col] = np.maximum(0, avg[col] - df_clean[col])
    else:
        PDA[col] = np.maximum(0, avg[col] - df_clean[col])
        NDA[col] = np.maximum(0, df_clean[col] - avg[col])

🔹 SP & SN

In [10]:
SP = PDA.dot(bobot)
SN = NDA.dot(bobot)

🔹 Normalisasi

In [11]:
NSP = SP / SP.max()
NSN = 1 - (SN / SN.max())

🔹 Skor akhir

In [12]:
AS = 0.5 * (NSP + NSN)

df_edas = df_clean.copy()
df_edas['EDAS_score'] = AS

ranking_edas = df_edas[['nomor_polis','EDAS_score']].sort_values(by='EDAS_score', ascending=False)
ranking_edas['Rank_EDAS'] = range(1, len(ranking_edas)+1)

print("TOP 10 EDAS")
ranking_edas.head(10)

TOP 10 EDAS


,nomor_polis,EDAS_score,Rank_EDAS
3215,POL-2721,1.000000,1
4120,POL-3623,0.999813,2
2340,POL-2125,0.999783,3
3042,POL-2515,0.999783,4
3966,POL-3436,0.999770,5
3821,POL-3296,0.999702,6
2338,POL-2125,0.999674,7
881,POL-0856,0.999624,8
4147,POL-3642,0.999617,9
2341,POL-2125,0.999348,10


METODE TOPSIS (TAMBAHAN)

🔹 Normalisasi

In [13]:
df_topsis = df_clean.copy()

norm = np.sqrt((df_topsis[kriteria]**2).sum())
df_topsis[kriteria] = df_topsis[kriteria] / norm

🔹 Pembobotan

In [14]:
df_topsis[kriteria] = df_topsis[kriteria] * bobot

🔹 Solusi ideal

In [15]:
ideal_pos = []
ideal_neg = []

for i, col in enumerate(kriteria):
    if benefit[i]:
        ideal_pos.append(df_topsis[col].max())
        ideal_neg.append(df_topsis[col].min())
    else:
        ideal_pos.append(df_topsis[col].min())
        ideal_neg.append(df_topsis[col].max())

ideal_pos = np.array(ideal_pos)
ideal_neg = np.array(ideal_neg)

🔹 Jarak & skor

In [16]:
D_pos = np.sqrt(((df_topsis[kriteria] - ideal_pos)**2).sum(axis=1))
D_neg = np.sqrt(((df_topsis[kriteria] - ideal_neg)**2).sum(axis=1))

df_topsis['TOPSIS_score'] = D_neg / (D_pos + D_neg)

ranking_topsis = df_topsis[['nomor_polis','TOPSIS_score']].sort_values(by='TOPSIS_score', ascending=False)
ranking_topsis['Rank_TOPSIS'] = range(1, len(ranking_topsis)+1)

print("TOP 10 TOPSIS")
ranking_topsis.head(10)

TOP 10 TOPSIS


,nomor_polis,TOPSIS_score,Rank_TOPSIS
3215,POL-2721,1.000000,1
4120,POL-3623,0.999993,2
3966,POL-3436,0.999991,3
2340,POL-2125,0.999989,4
3042,POL-2515,0.999989,5
3821,POL-3296,0.999989,6
881,POL-0856,0.999986,7
2338,POL-2125,0.999984,8
4147,POL-3642,0.999981,9
2341,POL-2125,0.999975,10


🔷 8. PERBANDINGAN HASIL

In [17]:
compare = ranking_saw.merge(ranking_edas, on='nomor_polis')
compare = compare.merge(ranking_topsis, on='nomor_polis')

compare = compare[['nomor_polis','Rank_SAW','Rank_EDAS','Rank_TOPSIS']]

🔹 TOP 10 PERBANDINGAN

In [18]:
compare_sorted = compare.sort_values(by='Rank_SAW')

print("TOP 10 PERBANDINGAN")
compare_sorted.head(10)

TOP 10 PERBANDINGAN


,nomor_polis,Rank_SAW,Rank_EDAS,Rank_TOPSIS
0,POL-0003,1,2197,2197
2,POL-0006,2,3534,4033
1,POL-0006,2,3534,3524
3,POL-0006,2,4034,3524
4,POL-0006,2,4034,4033
7,POL-0006,3,4034,3524
6,POL-0006,3,3534,4033
5,POL-0006,3,3534,3524
8,POL-0006,3,4034,4033
11,POL-0010,4,3559,903


🔹 ANALISIS KEMIRIPAN

In [19]:
same = (compare['Rank_SAW'] == compare['Rank_EDAS']).sum()
total = len(compare)

persentase_sama = (same / total) * 100
persentase_beda = 100 - persentase_sama

print("Jumlah Sama:", same)
print("Total Data:", total)
print("Persentase Sama:", persentase_sama, "%")
print("Persentase Berbeda:", persentase_beda, "%")

Jumlah Sama: 266
Total Data: 17872533
Persentase Sama: 0.0014883172967144612 %
Persentase Berbeda: 99.99851168270328 %


🔷 9. FORMAT TABEL SESUAI SOAL

In [20]:
tabel_akhir = compare_sorted.copy()
tabel_akhir['Urutan'] = range(1, len(tabel_akhir)+1)

tabel_akhir = tabel_akhir[['Urutan','nomor_polis','Rank_SAW','Rank_EDAS','Rank_TOPSIS']]

tabel_akhir.head(10)

,Urutan,nomor_polis,Rank_SAW,Rank_EDAS,Rank_TOPSIS
0,1,POL-0003,1,2197,2197
2,2,POL-0006,2,3534,4033
1,3,POL-0006,2,3534,3524
3,4,POL-0006,2,4034,3524
4,5,POL-0006,2,4034,4033
7,6,POL-0006,3,4034,3524
6,7,POL-0006,3,3534,4033
5,8,POL-0006,3,3534,3524
8,9,POL-0006,3,4034,4033
11,10,POL-0010,4,3559,903


🔷 10. EXPORT CSV (FULL DATA)

In [23]:
tabel_akhir.to_csv('hasil_perbandingan_full.csv', index=False)

from google.colab import files
files.download('hasil_perbandingan_full.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🔹 Pemrosesan Data
Dataset polis & klaim digabung menggunakan nomor_polis
Dilakukan pembersihan data (missing value)
Penentuan kriteria, bobot, dan tipe benefit/cost dibuat sama

🔹 SAW
Normalisasi berdasarkan min/max
Menghasilkan ranking berdasarkan skor preferensi

🔹 EDAS
Menggunakan rata-rata sebagai acuan
Menghitung PDA & NDA
Menghasilkan appraisal score

🔹 TOPSIS
Normalisasi vektor
Menghitung solusi ideal positif & negatif
Ranking berdasarkan kedekatan solusi ideal

🔹 Analisis
Dibandingkan hasil ranking SAW vs EDAS
Dihitung jumlah data dengan ranking sama
Didapatkan persentase kesamaan & perbedaan